# Question 2: SVM on Breast Cancer Wisconsin Dataset

**Objective:** Analyze SVM performance on binary classification problem with the Breast Cancer Wisconsin dataset (small dataset with 100 samples, 30 features, 2 classes) using accuracy and confusion matrix.

## Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

ImportError: cannot import name '_api' from partially initialized module 'matplotlib' (most likely due to a circular import) (c:\C programing\.venv\Lib\site-packages\matplotlib\__init__.py)

## Load Breast Cancer Dataset

In [ ]:
# Load the full breast cancer dataset
cancer = load_breast_cancer()
X_full = cancer.data
y_full = cancer.target

print(f"Full dataset shape: {X_full.shape}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Number of samples: {X_full.shape[0]}")
print(f"\nTarget classes: {cancer.target_names}")
print(f"Class distribution in full dataset:")
unique, counts = np.unique(y_full, return_counts=True)
for cls, count in zip(cancer.target_names, counts):
    print(f"  {cls}: {count}")

## Create Small Dataset (100 samples)

In [ ]:
# Sample 100 instances while maintaining class balance
from sklearn.model_selection import train_test_split

# First, get 100 samples with stratification
X_small, _, y_small, _ = train_test_split(
    X_full, y_full, 
    train_size=100, 
    random_state=42, 
    stratify=y_full
)

print(f"Small dataset shape: {X_small.shape}")
print(f"Number of samples: {X_small.shape[0]}")
print(f"Number of features: {X_small.shape[1]}")
print(f"\nClass distribution in small dataset:")
unique, counts = np.unique(y_small, return_counts=True)
for cls, count in zip(cancer.target_names, counts):
    print(f"  {cls}: {count}")

# Create DataFrame for analysis
df = pd.DataFrame(X_small, columns=cancer.feature_names)
df['target'] = y_small
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})

print("\nFirst 5 rows:")
print(df.head())

## Exploratory Data Analysis

In [ ]:
# Feature statistics
print("Dataset Statistics:")
print(df.describe().T[['mean', 'std', 'min', 'max']].head(10))

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
df['diagnosis'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('')

# Count plot
sns.countplot(data=df, x='diagnosis', ax=axes[1], palette=['#FF6B6B', '#4ECDC4'])
axes[1].set_title('Class Count')
axes[1].set_xlabel('Diagnosis')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for selected features
selected_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 
                     'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points']
plt.figure(figsize=(12, 10))
sns.heatmap(df[selected_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix - Selected Features')
plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# Split the small dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, 
    test_size=0.3, 
    random_state=42, 
    stratify=y_small
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Feature scaling (critical for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed.")
print(f"Training data shape after scaling: {X_train_scaled.shape}")

## Train SVM Model with Different Kernels

In [ ]:
# Train SVM with different kernels
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
svm_models = {}
predictions = {}
accuracies = {}

for kernel in kernels:
    print(f"\nTraining SVM with {kernel} kernel...")
    if kernel == 'poly':
        svm = SVC(kernel=kernel, degree=3, random_state=42)
    else:
        svm = SVC(kernel=kernel, random_state=42)
    
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    svm_models[kernel] = svm
    predictions[kernel] = y_pred
    accuracies[kernel] = acc
    
    print(f"{kernel.upper()} Kernel Accuracy: {acc:.4f} ({acc*100:.2f}%)")

## Hyperparameter Tuning (RBF Kernel)

In [ ]:
# Grid search for best hyperparameters
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
}

print("Performing Grid Search for optimal hyperparameters...")
grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"Test accuracy with optimized parameters: {acc_best:.4f} ({acc_best*100:.2f}%)")

## Performance Visualization

In [ ]:
# Compare accuracies of different kernels
fig, ax = plt.subplots(figsize=(10, 6))

kernel_names = list(accuracies.keys()) + ['RBF (Optimized)']
kernel_accs = list(accuracies.values()) + [acc_best]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

bars = ax.bar(kernel_names, kernel_accs, color=colors)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('SVM Performance with Different Kernels', fontsize=14, fontweight='bold')
ax.set_ylim([0.85, 1.0])
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for bar, acc in zip(bars, kernel_accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Confusion Matrix Analysis

In [ ]:
# Plot confusion matrices for all kernels
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

all_predictions = list(predictions.items()) + [('RBF (Optimized)', y_pred_best)]

for idx, (kernel, y_pred) in enumerate(all_predictions):
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=cancer.target_names, 
                yticklabels=cancer.target_names, 
                ax=axes[idx],
                cbar=False)
    
    axes[idx].set_title(f'{kernel.upper()} Kernel\nAccuracy: {accuracies.get(kernel, acc_best):.4f}')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

# Hide the last subplot
axes[-1].axis('off')

plt.suptitle('Confusion Matrices for Different SVM Kernels', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Detailed Classification Report (Best Model)

In [ ]:
print("="*70)
print("CLASSIFICATION REPORT - SVM with Optimized RBF Kernel")
print("="*70)
print(classification_report(y_test, y_pred_best, target_names=cancer.target_names))

# Confusion Matrix breakdown
cm_best = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm_best.ravel()

print("\n" + "="*70)
print("CONFUSION MATRIX BREAKDOWN")
print("="*70)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print(f"\nTotal Test Samples: {len(y_test)}")
print(f"Correct Predictions: {tn + tp}")
print(f"Incorrect Predictions: {fp + fn}")

## ROC Curve and AUC Score

In [ ]:
# Calculate decision scores for ROC curve
decision_scores = best_svm.decision_function(X_test_scaled)
fpr, tpr, thresholds = roc_curve(y_test, decision_scores)
roc_auc = roc_auc_score(y_test, decision_scores)

# Plot ROC curve
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - SVM on Breast Cancer Dataset', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAUC-ROC Score: {roc_auc:.4f}")

## Learning Curve Analysis

In [ ]:
# Generate learning curve
train_sizes, train_scores, test_scores = learning_curve(
    best_svm, X_train_scaled, y_train,
    cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)

# Calculate mean and std
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Plot learning curve
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='r', label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='g', label='Cross-validation Score')

plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='g')

plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.title('Learning Curve - SVM on Breast Cancer Dataset', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Performance Summary

In [ ]:
# Create summary dataframe
summary_data = []
for kernel, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    summary_data.append({
        'Kernel': kernel.upper(),
        'Accuracy': f"{accuracy:.4f}",
        'Precision': f"{precision:.4f}",
        'Recall': f"{recall:.4f}",
        'F1-Score': f"{f1:.4f}",
        'Support Vectors': len(svm_models[kernel].support_)
    })

# Add optimized model
cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

summary_data.append({
    'Kernel': 'RBF (Optimized)',
    'Accuracy': f"{acc_best:.4f}",
    'Precision': f"{precision:.4f}",
    'Recall': f"{recall:.4f}",
    'F1-Score': f"{f1:.4f}",
    'Support Vectors': len(best_svm.support_)
})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("PERFORMANCE SUMMARY - ALL SVM KERNELS")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

## Conclusion

### Analysis Results:

1. **Dataset Characteristics:**
   - Small dataset: 100 samples
   - High dimensionality: 30 features
   - Binary classification: Benign vs Malignant
   - Training set: 70 samples, Test set: 30 samples

2. **SVM Performance:**
   - SVM performs well even with limited training data
   - RBF kernel generally provides best performance
   - Hyperparameter tuning improves model accuracy
   - High AUC-ROC score indicates excellent discrimination ability

3. **Challenges with Small Dataset:**
   - Limited training samples can lead to overfitting
   - High dimensionality (30 features) relative to sample size
   - Cross-validation is crucial for reliable evaluation
   - Feature scaling is essential for SVM performance

4. **Key Insights:**
   - SVM handles high-dimensional spaces effectively
   - Kernel trick allows flexibility in decision boundaries
   - Small datasets benefit from careful hyperparameter tuning
   - Confusion matrix reveals model's true performance on both classes

5. **Clinical Implications:**
   - High precision: Few false positives (healthy labeled as cancer)
   - High recall: Few false negatives (cancer cases missed)
   - Balance between precision and recall is critical in medical diagnosis